# 🐍 Master Guide: Custom Exceptions & Enterprise Logging
> **Single-Source Revision Guide**: Covers domain-specific exception hierarchies, the `logging` framework, handlers, formatters, and exception groups (Python 3.11+).
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Production systems communicate failures through custom exception hierarchies and structured logging. The `logging` module offers hierarchical logger trees, severity levels, and automated log rotation.

---
## 🧠 2. CPython Architecture: The Logging Pipeline & Exception Groups

### The 4 Pillars of Python Logging
1. **Logger**: The entry point interface (`logger.info()`, `logger.error()`). Organized in a dot-separated hierarchy (e.g. `app.service.database`).
2. **Handler**: Directs log records to destinations (`StreamHandler` for console, `FileHandler` for disk, `RotatingFileHandler`).
3. **Formatter**: Dictates layout and structure (`%(asctime)s [%(levelname)s] %(message)s`).
4. **Filter**: Fine-grained gating logic deciding which records pass through.

### Exception Groups (`except*` in Python 3.11+)
Python 3.11 introduced `ExceptionGroup` allowing multiple concurrent exceptions to be raised and caught simultaneously using `except*`.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Creating Domain Exception Hierarchies
Always inherit from `Exception` (never `BaseException`) and establish a shared base exception for your package or domain.

In [ ]:
# Base domain exception
class BankingError(Exception):
    """Base exception for all banking operations."""
    pass

# Specific domain exceptions
class InsufficientFundsError(BankingError):
    def __init__(self, account_id, current_balance, amount_requested):
        super().__init__(
            f"Account '{account_id}' has ${current_balance:.2f}, "
            f"cannot withdraw ${amount_requested:.2f}!"
        )
        self.account_id = account_id
        self.current_balance = current_balance
        self.amount_requested = amount_requested

class AccountFrozenError(BankingError):
    pass

# Usage:
try:
    raise InsufficientFundsError("ACC-101", 45.0, 100.0)
except BankingError as e:
    print("Caught application-specific error:", e)
    if isinstance(e, InsufficientFundsError):
        print(f"Shortfall: ${e.amount_requested - e.current_balance:.2f}")

### Step 2: The `logging` Framework Basics
Severity levels: `DEBUG` (10) < `INFO` (20) < `WARNING` (30) < `ERROR` (40) < `CRITICAL` (50).

In [ ]:
import logging

# Configure root logger:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S"
)

logger = logging.getLogger("PaymentService")

logger.debug("This debug message won't show (level is INFO).")
logger.info("Payment service started successfully.")
logger.warning("High latency detected on payment gateway.")
logger.error("Transaction 889 failed: Timeout.")

---
## 🟡 4. Part 2: Intermediate Features

### Logging Tracebacks with `logger.exception()`
`logger.exception()` logs an `ERROR` level message and automatically captures the full stack trace.

In [ ]:
try:
    10 / 0
except ZeroDivisionError:
    logger.exception("Calculation failed unexpectedly:")

### Python 3.11+ Exception Groups (`except*`)
Handling multiple concurrent failures.

In [ ]:
eg = ExceptionGroup("Batch Job Failures", [
    ValueError("Invalid payload"),
    FileNotFoundError("Missing config"),
    ValueError("Negative price")
])

try:
    raise eg
except* ValueError as e:
    print(f"Handled ValueErrors: {e.exceptions}")
except* FileNotFoundError as e:
    print(f"Handled FileNotFoundError: {e.exceptions}")

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: `print()` vs `logging` in Production
- `print()` is synchronous, lacks log levels, lacks timestamps, and cannot be routed to rotating files or remote monitoring tools (Datadog/ELK).
- Always use `logging.getLogger(__name__)` in production modules!

In [ ]:
print("Rule: Never leave bare print() statements in library or backend code!")

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| Custom Exception                   | Inherit from Exception, NOT BaseException|
| logging.getLogger(__name__)        | Idiomatic way to instantiate loggers  |
| logger.exception()                 | Automatically includes full traceback |
| Log Severity                       | DEBUG < INFO < WARNING < ERROR < CRIT |
| ExceptionGroup (3.11+)             | Handled concurrently using except*    |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Programmer-Defined Custom Exceptions (`Development of Programmer-Defined Exceptions.txt`, `MulExceptions.py`)
> **Core Custom Exception Rules**:
> 1. Every programmer-defined exception class must inherit from `Exception` (or a subclass of `Exception`).
> 2. Use `raise <CustomExceptionClass>("error message")` to signal domain errors.
> 3. Catch custom exceptions specifically to trigger domain recovery logic.

In [ ]:
# Custom Exceptions: Banking & Validation Domain
class InsufficientFundsError(Exception):
    """Raised when withdrawal amount exceeds available balance."""
    pass

class NegativeAmountError(Exception):
    """Raised when transaction amount is zero or negative."""
    pass

class InvalidPinError(Exception):
    """Raised when user enters an incorrect PIN 3 times."""
    pass

class ATMController:
    def __init__(self, initial_balance=25000.0, pin=1234):
        self.balance = initial_balance
        self.pin = pin

    def withdraw(self, amount, entered_pin):
        if entered_pin != self.pin:
            raise InvalidPinError("Authentication Failed: Invalid ATM PIN entered!")
        if amount <= 0:
            raise NegativeAmountError(f"Invalid Amount: Rs. {amount}. Amount must be positive!")
        if amount > self.balance:
            raise InsufficientFundsError(
                f"Declined: Requested Rs. {amount:,.2f}, Available: Rs. {self.balance:,.2f}"
            )
        self.balance -= amount
        print(f"SUCCESS: Dispensed Rs. {amount:,.2f}. Balance: Rs. {self.balance:,.2f}")

atm = ATMController()

# Testing custom exceptions
for test_amt, test_pin in [(5000, 1234), (-500, 1234), (40000, 1234), (1000, 9999)]:
    try:
        atm.withdraw(test_amt, test_pin)
    except (InsufficientFundsError, NegativeAmountError, InvalidPinError) as e:
        print(f"[CUSTOM EXCEPTION HANDLED] {type(e).__name__}: {e}")